# Apache Beam Lab 4: I/O Connectors

## Overview
This lab covers Apache Beam's I/O connectors for reading from and writing to various data sources. Understanding I/O is crucial for building production-ready pipelines.

## Learning Objectives
- Read from various file formats (CSV, JSON, Parquet)
- Write output to different sinks
- Understand file-based I/O patterns
- Work with text files and custom formats
- Handle I/O errors and retries

## Prerequisites
- Completed Lab 3: Core Transforms
- Sample data available from Lab 0

## File-based I/O
Apache Beam provides built-in I/O connectors for common file formats.

In [ ]:
import apache_beam as beam
import os

# Text file I/O example
def text_io_example():
    # Create sample text file
    sample_file = '/tmp/sample_text.txt'
    with open(sample_file, 'w') as f:
        f.write('hello world\n')
        f.write('apache beam\n')
        f.write('data processing\n')
    
    with beam.Pipeline() as pipeline:
        (
            pipeline
            | 'Read text' >> beam.io.ReadFromText(sample_file)
            | 'Process' >> beam.Map(str.upper)
            | 'Write output' >> beam.io.WriteToText('/tmp/output_text')
        )
    
    print("Text I/O example completed")

print("File-based I/O: Read from and write to various file formats")

## Exercise 1: CSV File Processing
Create a pipeline that reads CSV files, processes the data, and writes results to a new CSV file.

In [ ]:
# Your code here to process CSV files
# 1. Read the sample sales CSV file
# 2. Calculate total revenue per product
# 3. Write results to a new CSV file

def csv_processing_pipeline():
    sales_file = '/home/ramdov/projects/beam-code-practice/data/sample_sales.csv'
    output_file = '/tmp/sales_summary.csv'
    
    if not os.path.exists(sales_file):
        print("Sales data not found. Run Lab 0 first.")
        return
    
    # Read CSV manually for processing
    import pandas as pd
    df = pd.read_csv(sales_file)
    data = df.to_dict('records')
    
    with beam.Pipeline() as pipeline:
        results = (
            pipeline
            | 'Create data' >> beam.Create(data)
            | 'Calculate totals' >> beam.Map(
                lambda x: (x['product_id'], x['quantity'] * x['price']))
            | 'Sum by product' >> beam.CombinePerKey(sum)
            | 'Format for CSV' >> beam.Map(
                lambda x: f"{x[0]},{x[1]:.2f}")
            | 'Write to file' >> beam.io.WriteToText(output_file, shard_name_template='')
        )
    
    # Add header
    with open(output_file, 'r') as f:
        content = f.read()
    with open(output_file, 'w') as f:
        f.write('product_id,total_revenue\n')
        f.write(content)
    
    print(f"CSV processing completed. Results written to {output_file}")

print("Processing CSV files...")
csv_processing_pipeline()

## Exercise 2: JSON File Processing
Create a pipeline that reads JSON data, transforms it, and writes the results.

In [ ]:
# Your code here to process JSON files
# 1. Create sample JSON data
# 2. Read and process the JSON
# 3. Write transformed data to a new JSON file

import json

def json_processing_pipeline():
    # Create sample JSON file
    sample_json = '/tmp/sample_data.json'
    sample_data = [
        {"id": 1, "name": "Alice", "score": 85},
        {"id": 2, "name": "Bob", "score": 92},
        {"id": 3, "name": "Charlie", "score": 78}
    ]
    
    with open(sample_json, 'w') as f:
        json.dump(sample_data, f)
    
    with beam.Pipeline() as pipeline:
        (
            pipeline
            | 'Create JSON data' >> beam.Create(sample_data)
            | 'Add grade' >> beam.Map(
                lambda x: {
                    **x,
                    'grade': 'A' if x['score'] >= 90 else 'B' if x['score'] >= 80 else 'C'
                })
            | 'Convert to JSON string' >> beam.Map(json.dumps)
            | 'Write to file' >> beam.io.WriteToText('/tmp/processed_data', shard_name_template='')
        )
    
    # Format as JSON array
    with open('/tmp/processed_data', 'r') as f:
        lines = f.readlines()
    
    json_array = '[' + ','.join([line.strip() for line in lines]) + ']'
    with open('/tmp/processed_data.json', 'w') as f:
        f.write(json_array)
    
    print("JSON processing completed. Results written to /tmp/processed_data.json")

print("Processing JSON files...")
json_processing_pipeline()

## Exercise 3: Multiple File Processing
Create a pipeline that processes multiple files and combines the results.

In [ ]:
# Your code here to process multiple files
# 1. Create multiple sample files
# 2. Read from all files using a pattern
# 3. Combine and process the data
# 4. Write unified results

def multi_file_processing():
    # Create sample files
    for i in range(3):
        filename = f'/tmp/data_part_{i}.txt'
        with open(filename, 'w') as f:
            for j in range(5):
                f.write(f'file_{i}_line_{j}\n')
    
    with beam.Pipeline() as pipeline:
        (
            pipeline
            | 'Read multiple files' >> beam.io.ReadFromText('/tmp/data_part_*.txt')
            | 'Extract file number' >> beam.Map(
                lambda x: (x.split('_')[1], 1))
            | 'Count per file' >> beam.CombinePerKey(sum)
            | 'Format' >> beam.Map(lambda x: f"File {x[0]}: {x[1]} lines")
            | 'Print' >> beam.Map(print)
        )
    
    print("Multi-file processing completed")

print("Processing multiple files...")
multi_file_processing()

## Summary

In this lab, you have:
- Read from various file formats (Text, CSV, JSON)
- Written output to different file formats
- Processed multiple files using patterns
- Understood file-based I/O patterns
- Applied I/O operations to real scenarios

## Key Takeaways
- Apache Beam provides built-in I/O for common formats
- File patterns enable processing multiple files
- I/O operations are integrated into pipeline DAG
- Error handling and retries are built-in
- Output formatting depends on downstream requirements

## Next Steps

Proceed to [Lab 5: Windowing and Triggers](lab-05-windowing-triggers.md) to learn about processing streaming data with time-based windows.